In [1]:
# Cell 1: Install Required Packages

!pip install ASE
!pip install mace-torch ase rdkit weas-widget

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 25.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.7/387.7 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.3/345.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.1/453.1 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 79.3 MB/s eta 0:00:00
  Created wheel for python-hostlist: filename=python_hostlist-2.3.0-py3-none-any.whl size=39449 sha256=88e7eb93f48686356139bdbd219a51967ee45eecd79f10590caedfe407533b0d
  Stored in directory: /root/.cache/pip/wheels/02/e4/34/75fc0cd5b7889d8cc

In [2]:
# Cell 2: Import Required Libraries

import numpy as np
from ase.optimize import QuasiNewton
from ase.thermochemistry import IdealGasThermo
from ase.vibrations import Vibrations
from ase.units import kJ, mol
from ase import Atoms
from ase.build import bulk, molecule
import pandas as pd
from mace.calculators import mace_mp, mace_off

from rdkit import Chem
from rdkit.Chem import AllChem

/usr/local/lib/python3.12/dist-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


In [3]:
# Cell 3: Load MACE-OFF

print("Loading MACE-OFF (medium model)...")
calc_mol = mace_off(model="small", default_dtype="float64")
print("MACE-OFF loaded.")

Loading MACE-OFF (medium model)...
The model is distributed under the Academic Software License (ASL) license, see https://github.com/gabor1/ASL 
 To use the model you accept the terms of the license.
ASL is based on the Gnu Public License, but does not permit commercial use
Downloading: 100.0% (7.0 MB / 7.0 MB)
Cached MACE model to /root/.cache/mace/MACE-OFF23_small.model
Using MACE-OFF23 MODEL for MACECalculator with /root/.cache/mace/MACE-OFF23_small.model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/usr/local/lib/python3.12/dist-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


MACE-OFF loaded.


In [4]:
# Cell 4: Calculate Methyl Nitrite (CH3ONO) Thermodynamic Properties

atoms_CH3ONO = molecule('CH3ONO') # Picks Molecule from g2 list (Quickly pick common molecules)
atoms_CH3ONO.calc = calc_mol # Calls to MACE-OFF to be used
dyn = QuasiNewton(atoms_CH3ONO, logfile=None)
dyn.run(fmax=0.01) # Finding energy minimum
potentialenergy_CH3ONO = atoms_CH3ONO.get_potential_energy() # Get potential energy

vib = Vibrations(atoms_CH3ONO, name='ch3ono_vib')
vib.clean()
vib.run()
vib_energies = vib.get_energies() # Gets vibrational energy based on molecule geometry
vib_energies = np.array([e.real for e in vib_energies if e.real > 0.01]) # Filters out imaginary numbers and very low frequencies

# Takes inputs of vibrational energies, potential energy, and geometry to compute thermodynamic properties
thermo = IdealGasThermo(
    vib_energies=vib_energies,
    potentialenergy=potentialenergy_CH3ONO,
    atoms=atoms_CH3ONO,
    geometry='nonlinear', # Linear (Straight Line) or Nonlinear (Bent in any way)
    symmetrynumber=1, # How many times you can rotate the molecule and get the same configuration
    spin=0, # 0.5 for each unpaired electrons
)


# Records Thermodynamic Property Data at 6 Temperatures at 1 atmosphere and Displays it
temps = [298.15, 400, 500, 600, 700, 800]
P = 101325.

records = []
for T in temps:
    H = thermo.get_enthalpy(T, verbose=False)
    S = thermo.get_entropy(T, P, verbose=False)
    G = thermo.get_gibbs_energy(T, P, verbose=False)
    records.append({"T (K)": T, "H (eV)": H, "S (eV/K)": S, "G (eV)": G})

df1 = pd.DataFrame(records)
display(df1)

,T (K),H (eV),S (eV/K),G (eV)
0,298.15,-6669.853239,0.002861,-6670.706377
1,400.00,-6669.779623,0.003073,-6671.008786
2,500.00,-6669.695084,0.003261,-6671.325623
3,600.00,-6669.600004,0.003434,-6671.660501
4,700.00,-6669.496061,0.003594,-6672.012022
5,800.00,-6669.384683,0.003743,-6672.378966


In [5]:
# Cell 5: Calculate Methyl Nitrite (CH3ONO) Thermodynamic Properties without using g2 list
# Build the Molecule using SMILES

smiles = 'CON=O' # SMILES for Methyl Nitrite (CH3ONO)
seed = 42

mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol) # Adds explicit hydrogens to the molecule
AllChem.EmbedMolecule(mol, randomSeed=seed) # Randomly places atoms at correct distances from each other
AllChem.MMFFOptimizeMolecule(mol) # Optimizes the geometry using the MMFF94 classical force field (Starting Position for MACE-OFF optimization))
conf = mol.GetConformer()
symbols = [a.GetSymbol() for a in mol.GetAtoms()]
positions = conf.GetPositions()

atoms_CH3ONO = Atoms(symbols=symbols, positions=positions)

# Same as last code except atoms_CH3ONO is defined instead of being called from g2 list
atoms_CH3ONO.calc = calc_mol # Calls to MACE-OFF to be used
dyn = QuasiNewton(atoms_CH3ONO, logfile=None)
dyn.run(fmax=0.01) # Finding energy minimum
potentialenergy_CH3ONO = atoms_CH3ONO.get_potential_energy() # Get potential energy

vib = Vibrations(atoms_CH3ONO, name='ch3ono_vib')
vib.clean()
vib.run()
vib_energies = vib.get_energies() # Gets vibrational energy based on molecule geometry
vib_energies = np.array([e.real for e in vib_energies if e.real > 0.01]) # Filters out imaginary numbers and very low frequencies

# Takes inputs of vibrational energies, potential energy, and geometry to compute thermodynamic properties
thermo = IdealGasThermo(
    vib_energies=vib_energies,
    potentialenergy=potentialenergy_CH3ONO,
    atoms=atoms_CH3ONO,
    geometry='nonlinear', # Linear (Straight Line) or Nonlinear (Bent in any way)
    symmetrynumber=1, # How many times you can rotate the molecule and get the same configuration
    spin=0, # 0.5 for each unpaired electrons
)


# Records Thermodynamic Property Data at 6 Temperatures at 1 atmosphere and Displays it
temps = [298.15, 400, 500, 600, 700, 800]
P = 101325.

records = []
for T in temps:
    H = thermo.get_enthalpy(T, verbose=False)
    S = thermo.get_entropy(T, P, verbose=False)
    G = thermo.get_gibbs_energy(T, P, verbose=False)
    records.append({"T (K)": T, "H (eV)": H, "S (eV/K)": S, "G (eV)": G})

df2 = pd.DataFrame(records)
display(df2)

,T (K),H (eV),S (eV/K),G (eV)
0,298.15,-6669.877876,0.002949,-6670.757160
1,400.00,-6669.803150,0.003164,-6671.068716
2,500.00,-6669.718645,0.003352,-6671.394665
3,600.00,-6669.624145,0.003524,-6671.738591
4,700.00,-6669.521014,0.003683,-6672.099041
5,800.00,-6669.410517,0.003830,-6672.474791


In [6]:
# Cell 6: Side by Side Comparison

print("Molecule Called from g2 List")
display(df1)

print("")
print("Molecule Actually Being Build")
display(df2)

Molecule Called from g2 List


,T (K),H (eV),S (eV/K),G (eV)
0,298.15,-6669.853239,0.002861,-6670.706377
1,400.00,-6669.779623,0.003073,-6671.008786
2,500.00,-6669.695084,0.003261,-6671.325623
3,600.00,-6669.600004,0.003434,-6671.660501
4,700.00,-6669.496061,0.003594,-6672.012022
5,800.00,-6669.384683,0.003743,-6672.378966



Molecule Actually Being Build


,T (K),H (eV),S (eV/K),G (eV)
0,298.15,-6669.877876,0.002949,-6670.757160
1,400.00,-6669.803150,0.003164,-6671.068716
2,500.00,-6669.718645,0.003352,-6671.394665
3,600.00,-6669.624145,0.003524,-6671.738591
4,700.00,-6669.521014,0.003683,-6672.099041
5,800.00,-6669.410517,0.003830,-6672.474791
